<a href="https://colab.research.google.com/github/Nidhijain0602/CareLoop-Clinics-No-Show-Guard/blob/main/04_genai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Part 4 — GenAI: turn call-centre notes into structured reasons, then recommend**

Task 4.1 — One validated call

In [2]:
from google.colab import files
import pandas as pd
import json

uploaded = files.upload()

filename = list(uploaded.keys())[0]
notes_df = pd.read_csv(filename)

print("Dataset shape:", notes_df.shape)
print(notes_df.head(10).to_string(index=False))

Saving followup_notes.csv to followup_notes (1).csv
Dataset shape: (30, 3)
note_id appointment_id                                                                                                                     note_text
   N001          A0872         Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
   N002          A0897   Lives far from the clinic and could not find any transport that morning. Says the clinic is too far for a routine check-up.
   N003          A0526 Patient felt the consultation fee was too high this month and decided to skip. Asked if there is any discount for follow-ups.
   N004          A0926                                                                                                                           NaN
   N005          A0534                                            Had high fever and body ache on the day, stayed in bed. Will visit once recovered.
   N006          A0507         

Task 4.2 — Validate and process the 10 notes

In [3]:

note_n001 = notes_df.loc[
    notes_df["note_id"] == "N001", "note_text"
].iloc[0]

print("Note N001:")
print(note_n001)

Note N001:
Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.


In [4]:
!pip install -q google-genai pydantic


In [ ]:
from google import genai
from google.genai import types
import json

API_KEY = input("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)

del API_KEY

MODEL_NAME = "gemini-3.1-flash-lite"

print("Gemini client configured.")


In [11]:
ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

SYSTEM_TEXT = (
    "You are a careful healthcare operations analyst. "
    "Classify missed appointment reasons using only "
    "the evidence in each follow-up note. "
    "Do not invent details."
)

def analyse_note(note_text):
    prompt = f"""
Context: CareLoop Clinics wants to understand
why patients missed outpatient appointments.

Role: You are a patient follow-up analyst.

Action: Read the note and classify its main reason.
Choose exactly one root_cause from:
{", ".join(ALLOWED_CAUSES)}.

If the reason is ambiguous, choose 'other'.
Do not guess.

Tone: Professional, neutral and concise.

Format: Return JSON with exactly four fields:
root_cause, confidence, summary, recommended_action.
Confidence must be between 0 and 1.
Summary must contain at most 25 words.

Patient follow-up note:
{note_text}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1500,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    return json.loads(response.text)

In [15]:
note_n001 = notes_df.loc[
    notes_df["note_id"] == "N001",
    "note_text"
].iloc[0]

result = analyse_note(note_n001)

print(json.dumps(result, indent=2))
print("Root cause:", result["root_cause"])

{
  "root_cause": "forgot_or_no_reminder",
  "confidence": 1.0,
  "summary": "Patient missed the appointment because they did not receive a reminder and subsequently forgot the scheduled visit.",
  "recommended_action": "Verify patient contact information and ensure automated reminder system is functioning correctly for future appointments."
}
Root cause: forgot_or_no_reminder


**Task 4.2 — Validate and process the 10 notes**

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def validate_insight(data):
    validated = FollowupInsight(**data)

    if validated.root_cause not in ALLOWED_CAUSES:
        raise ValueError(
            f"Invalid root cause: {validated.root_cause}"
        )

    if len(validated.summary.split()) > 25:
        raise ValueError("Summary exceeds 25 words")

    return validated

In [16]:

import time

records = []
api_calls = 1  # N001 was already called in Task 4.1

for _, row in notes_df.head(10).iterrows():
    note_id = row["note_id"]
    note_text = row["note_text"]

    record = {
        "note_id": note_id,
        "appointment_id": row["appointment_id"],
        "root_cause": None,
        "confidence": None,
        "summary": None,
        "recommended_action": None,
        "validation_status": None,
        "raw_response": ""
    }

    if pd.isna(note_text) or str(note_text).strip() == "":
        record.update({
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank"
        })

    else:
        try:
            if note_id == "N001":
                data = result
                raw_text = json.dumps(result)
            else:
                time.sleep(13)
                data = analyse_note(note_text)
                api_calls += 1
                raw_text = json.dumps(data)

            record["raw_response"] = raw_text

            for field in [
                "root_cause", "confidence",
                "summary", "recommended_action"
            ]:
                record[field] = data.get(field)

            validate_insight(data)
            record["validation_status"] = "valid"

        except Exception as error:
            record["validation_status"] = "invalid"
            print(f"{note_id} failed: {error}")

    records.append(record)

notes_analysed = pd.DataFrame(records)

print(notes_analysed[
    ["note_id", "root_cause", "validation_status"]
].to_string(index=False))

print("\nTotal API calls so far:", api_calls)

N001 failed: name 'validate_insight' is not defined
N002 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
N003 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
N005 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
N006 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
N007 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. 

In [17]:
notes_analysed.to_csv(
    "notes_analysed.csv",
    index=False
)

print("Shape:", notes_analysed.shape)
print(notes_analysed["validation_status"].value_counts())
print("API calls:", api_calls)

from google.colab import files
files.download("notes_analysed.csv")

Shape: (10, 8)
validation_status
invalid          9
skipped_blank    1
Name: count, dtype: int64
API calls: 3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:

from google.colab import files
import pandas as pd
import json

uploaded = files.upload()

print("Uploaded files:", list(uploaded.keys()))

Saving notes_analysed.csv to notes_analysed (1).csv
Uploaded files: ['notes_analysed (1).csv']


In [21]:
notes_filename = [
    name for name in uploaded
    if name != "notes_analysed.csv"
][0]

notes_df = pd.read_csv(notes_filename)
notes_analysed = pd.read_csv("notes_analysed.csv")

print("Original notes:", notes_df.shape)
print("Analysed notes:", notes_analysed.shape)

print(notes_analysed[
    ["note_id", "root_cause", "validation_status"]
].to_string(index=False))


Original notes: (10, 8)
Analysed notes: (10, 8)
note_id            root_cause validation_status
   N001 forgot_or_no_reminder           invalid
   N002                   NaN           invalid
   N003                   NaN           invalid
   N004                 other     skipped_blank
   N005                   NaN           invalid
   N006                   NaN           invalid
   N007                   NaN           invalid
   N008                   NaN           invalid
   N009                 other           invalid
   N010      work_or_schedule           invalid


In [24]:

import time
note_n010 = notes_df.loc[
    notes_df["note_id"] == "N010",
    "note_text"
].iloc[0]

time.sleep(13)

try:
    result_n010 = analyse_note(note_n010)
    validate_insight(result_n010)

    idx = notes_analysed.index[
        notes_analysed["note_id"] == "N010"
    ][0]

    for field in [
        "root_cause", "confidence",
        "summary", "recommended_action"
    ]:
        notes_analysed.at[idx, field] = result_n010.get(field)

    notes_analysed.at[idx, "validation_status"] = "valid"
    notes_analysed.at[idx, "raw_response"] = json.dumps(result_n010)

    print("N010 successfully processed!")
    print(result_n010)

except Exception as error:
    print("Retry failed:", error)

KeyError: 'note_text'

In [23]:
print(notes_df.columns.tolist())

['note_id', 'appointment_id', 'root_cause', 'confidence', 'summary', 'recommended_action', 'validation_status', 'raw_response']


In [25]:
print("Gemini function:", callable(analyse_note))
print("Validation function:", callable(validate_insight))

print("Original notes:", notes_df.shape)
print("Analysed notes:", notes_analysed.shape)

Gemini function: True


NameError: name 'validate_insight' is not defined

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()


from google.colab import files
import pandas as pd

uploaded = files.upload()

Saving notes_analysed.csv to notes_analysed (2).csv


In [ ]:
original_filename = [
    name for name in uploaded.keys()
    if name.startswith("ecWvPsdV2FNmRZAa")
][0]

notes_df = pd.read_csv(original_filename)

notes_analysed = pd.read_csv("notes_analysed.csv")

print("Original notes:", notes_df.shape)
print("Analysed notes:", notes_analysed.shape)

print("\nValidation results:")
print(notes_analysed[
    ["note_id", "root_cause", "validation_status"]
].to_string(index=False))

In [ ]:
print("Gemini:", callable(analyse_note))
print("Validation:", callable(validate_insight))
print("Original notes:", notes_df.shape)
print("Analysed notes:", notes_analysed.shape)

In [ ]:
import time
import json

note_n010 = notes_df.loc[
    notes_df["note_id"] == "N010",
    "note_text"
].iloc[0]

time.sleep(13)

try:
    result_n010 = analyse_note(note_n010)

    validate_insight(result_n010)

    idx = notes_analysed.index[
        notes_analysed["note_id"] == "N010"
    ][0]

    for field in [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]:
        notes_analysed.at[idx, field] = result_n010.get(field)

    notes_analysed.at[idx, "validation_status"] = "valid"
    notes_analysed.at[idx, "raw_response"] = json.dumps(result_n010)

    print("N010 successfully processed!")
    print(json.dumps(result_n010, indent=2))

except Exception as error:
    print("Retry failed:", error)

In [ ]:

print(notes_analysed[
    ["note_id", "root_cause", "validation_status"]
].to_string(index=False))

print("\nValidation counts:")
print(notes_analysed["validation_status"].value_counts())


In [ ]:
notes_analysed.to_csv(
    "notes_analysed.csv",
    index=False
)

from google.colab import files
files.download("notes_analysed.csv")

print("Updated CSV downloaded successfully!")

Task 4.3 — Break it on purpose

In [19]:

from pydantic import ValidationError

# Test 1: Confidence is above the allowed maximum
bad_response_1 = {
    "root_cause": "work_or_schedule",
    "confidence": 1.7,
    "summary": "Patient was busy at work",
    "recommended_action": "Offer flexible appointment hours"
}

# Test 2: Missing recommended_action
bad_response_2 = {
    "root_cause": "distance_or_transport",
    "confidence": 0.8,
    "summary": "Patient had transport difficulties"
}

for i, bad_response in enumerate(
    [bad_response_1, bad_response_2],
    start=1
):
    try:
        FollowupInsight(**bad_response)
        print(f"Test {i}: Unexpectedly passed")

    except ValidationError as error:
        print(f"\nTest {i}: Validation correctly failed")
        print(error)

NameError: name 'FollowupInsight' is not defined

In [ ]:

def analyse_note_v1(note_text):

    prompt = f"Classify this note: {note_text}"

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1500,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    return json.loads(response.text)


print("V1 basic prompt function is ready!")

In [ ]:


selected_notes = notes_df[
    notes_df["note_id"].isin(["N007", "N009"])
]

print(selected_notes[
    ["note_id", "note_text"]
].to_string(index=False))


In [ ]:

import time
import pandas as pd

comparison_records = []
comparison_api_calls = 0

for note_id in ["N007", "N009"]:

    note_text = notes_df.loc[
        notes_df["note_id"] == note_id,
        "note_text"
    ].iloc[0]

    # Test both prompt versions
    for prompt_name, function in [
        ("V1 Basic", analyse_note_v1),
        ("V2 CRAFT", analyse_note)
    ]:

        # Pause between API requests
        time.sleep(13)

        try:
            comparison_api_calls += 1
            response = function(note_text)

            root_cause = response.get(
                "root_cause", ""
            )

            is_allowed = (
                root_cause in ALLOWED_CAUSES
            )

            comparison_records.append({
                "Note": note_id,
                "Prompt": prompt_name,
                "Root Cause": root_cause,
                "Allowed": is_allowed
            })

        except Exception as error:

            comparison_records.append({
                "Note": note_id,
                "Prompt": prompt_name,
                "Root Cause": "",
                "Allowed": False
            })

            print(
                f"Error for {note_id} "
                f"({prompt_name}): {error}"
            )

prompt_comparison = pd.DataFrame(
    comparison_records
)

print("\nPROMPT COMPARISON RESULTS")
print(prompt_comparison.to_string(index=False))

print("\nComparison API calls:",
      comparison_api_calls)


**Task 4.3 — Prompt Comparison Findings**

The basic prompt (V1) produced zero valid root-cause labels out of two responses. For N007, it returned "Patient forgetfulness and lack of appointment reminder"; for N009, it returned "Communication failure". Both were outside the six permitted categories.

The CRAFT prompt (V2) produced two valid labels out of two responses: forgot_or_no_reminder for N007 and other for N009.

V2 is recommended for deployment because it explicitly defines the permitted categories and instructs the model to avoid unsupported guesses. However, this small experiment measures classification-format compliance rather than overall accuracy.

In [ ]:

api_call_breakdown = {
    "Task 4.1 N001": 1,
    "Task 4.2 batch": 8,
    "N010 retry": 1,
    "Task 4.3 prompt comparison": comparison_api_calls
}

total_api_calls = sum(api_call_breakdown.values())

print("API CALL BREAKDOWN")
for activity, count in api_call_breakdown.items():
    print(f"{activity}: {count}")

print("\nTOTAL API CALLS:", total_api_calls)
print("Under 25 calls:", total_api_calls < 25)


In [ ]:

from google.colab import files
import pandas as pd

uploaded_appointments = files.upload()

appointments_filename = list(
    uploaded_appointments.keys()
)[0]

appointments_df = pd.read_csv(
    appointments_filename
)

print("Appointments shape:", appointments_df.shape)
print("Analysed notes shape:", notes_analysed.shape)

In [ ]:
merged_notes = notes_analysed.merge(
    appointments_df,
    on="appointment_id",
    how="left",
    validate="one_to_one"
)

print("Merged dataset shape:", merged_notes.shape)
print("Matched appointments:",
      merged_notes["clinic_id"].notna().sum())

assert len(merged_notes) == 10
assert merged_notes["clinic_id"].notna().all()


In [ ]:

reason_reminder_table = pd.crosstab(
    merged_notes["root_cause"],
    merged_notes["reminder_sent"]
)

print("ROOT CAUSE VS REMINDER SENT")
print(reason_reminder_table)

print("\nRoot cause counts:")
print(merged_notes["root_cause"].value_counts())

print("\nAppointments without reminders:",
      (merged_notes["reminder_sent"] == "No").sum())

In [ ]:

print("Total analysed notes:", len(merged_notes))

print("Missing appointment matches:",
      merged_notes["clinic_id"].isna().sum())

print("Valid AI classifications:",
      (notes_analysed["validation_status"] == "valid").sum())

print("Skipped blank notes:",
      (notes_analysed["validation_status"] == "skipped_blank").sum())

In [ ]:


print("ROOT CAUSE VS REMINDER SENT")
print(reason_reminder_table.to_string())

print("\nROOT CAUSE COUNTS")
print(merged_notes["root_cause"].value_counts())

print("\nAPPOINTMENTS WITHOUT REMINDERS")
print((merged_notes["reminder_sent"] == "No").sum())